In [ ]:
import os, time, json, math
import pandas as pd
import numpy as np
from typing import Dict, List, Tuple, Any, Optional
from pathlib import Path

from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langchain_community.callbacks import get_openai_callback

from PyDI.entitymatching.blocking import (
    TokenBlocker,
    StandardBlocker, 
    SortedNeighbourhoodBlocker,
    EmbeddingBlocker
)
from PyDI.entitymatching.comparators import StringComparator, NumericComparator
from PyDI.entitymatching.feature_extraction import FeatureExtractor
from PyDI.entitymatching.evaluation import EntityMatchingEvaluator


class BlockingTestSetAgent:
    """
    LLM-driven agent that creates blocking evaluation test sets.
    The LLM decides which columns to compare and which similarity functions to use.
    """
    
    BLOCKER_TYPES = {
        "token": "TokenBlocker - blocks on shared tokens in a text column",
        "standard": "StandardBlocker - blocks on exact equality of column values",
        "sorted_neighbourhood": "SortedNeighbourhoodBlocker - sliding window over sorted key",
        "embedding": "EmbeddingBlocker - semantic similarity via embeddings"
    }
    
    # Available similarity functions the LLM can choose from
    SIMILARITY_FUNCTIONS = {
        "jaro_winkler": "String similarity using Jaro-Winkler (good for names, titles) - returns 0-1",
        "levenshtein": "Edit distance normalized to 0-1 (good for short strings, codes)",
        "exact_match": "Returns 1 if exactly equal (case-insensitive), 0 otherwise",
        "token_overlap": "Jaccard similarity of word tokens (good for long text with shared words)",
        "numeric_diff": "1 - |a-b|/max(a,b) for numbers (good for years, durations, prices)",
        "geo_distance": "Requires lat/lon - returns 1 if within threshold, decays to 0",
        "contains": "Returns 1 if one string contains the other, 0 otherwise"
    }
    
    def __init__(
        self,
        llm,
        output_dir: str = "output/testset-generation",
        n_samples: int = 100,
        checkpoint_interval: int = 10,
        verbose: bool = True,
        target_match_pct: float = 0.20,
        max_resample_rounds: int = 5,
        resample_batch_size: int = 20,
        max_candidates: int = 500000,
        min_token_len: int = 3
    ):
        self.llm = llm
        self.output_dir = output_dir
        self.n_samples = n_samples
        self.checkpoint_interval = checkpoint_interval
        self.verbose = verbose
        self._schema_analysis = None
        
        self.target_match_pct = target_match_pct
        self.max_resample_rounds = max_resample_rounds
        self.resample_batch_size = resample_batch_size
        self.max_candidates = max_candidates
        self.min_token_len = min_token_len
        
        self._reserve_high_candidates = None
        self._reserve_low_candidates = None
        self._reserve_mid_candidates = None
        self._blocker_instance = None
        
        # Store original data for generating random negatives
        self._df_a = None
        self._df_b = None
        self._id_col_a = None
        self._id_col_b = None
        self._all_candidate_ids = None
        
        # Token usage tracking
        self.token_usage = {
            'prompt_tokens': 0,
            'completion_tokens': 0,
            'total_tokens': 0,
            'total_cost': 0.0
        }
        
        os.makedirs(self.output_dir, exist_ok=True)
    
    def _analyze_schema_with_llm(self, df_a: pd.DataFrame, df_b: pd.DataFrame, sample_rows: int = 3) -> Dict[str, Any]:
        """Use LLM to analyze both datasets and determine the matching strategy with similarity functions."""
        def get_column_info(df, name):
            info = []
            for col in df.columns:
                dtype = str(df[col].dtype)
                non_null = df[col].notna().sum()
                unique = df[col].nunique()
                sample_vals = df[col].dropna().head(sample_rows).tolist()
                sample_str = ", ".join([str(v)[:50] for v in sample_vals])
                info.append(f"  - {col} (dtype={dtype}, non_null={non_null}, unique={unique}): [{sample_str}]")
            return f"{name} columns:\n" + "\n".join(info)
        
        schema_a = get_column_info(df_a, "Dataset A")
        schema_b = get_column_info(df_b, "Dataset B")
        blocker_descriptions = "\n".join([f"  - {k}: {v}" for k, v in self.BLOCKER_TYPES.items()])
        similarity_descriptions = "\n".join([f"  - {k}: {v}" for k, v in self.SIMILARITY_FUNCTIONS.items()])
        
        system_prompt = """You are an expert data engineer specializing in entity matching and record linkage.

Your task is to analyze two datasets and design a similarity scoring strategy. Think carefully about:
1. What type of real-world entity these records represent
2. Which columns are the MOST IMPORTANT for determining if two records are the same entity
3. Which columns can be IGNORED (IDs, metadata, redundant info)
4. How to weight the important columns based on their discriminative power

Respond with your reasoning followed by the JSON configuration."""

        human_prompt = f"""Analyze these two datasets for entity matching:

{schema_a}

{schema_b}

AVAILABLE BLOCKING STRATEGIES:
{blocker_descriptions}

AVAILABLE SIMILARITY FUNCTIONS:
{similarity_descriptions}

THINK STEP BY STEP:

1. ENTITY TYPE: What real-world thing do these records represent? (e.g., restaurants, songs, people, products)

2. KEY IDENTIFYING COLUMNS: Which 2-4 columns are MOST important for matching? Consider:
   - Primary identifiers (names, titles) → high weight (0.3-0.5)
   - Secondary identifiers (addresses, artists, dates) → medium weight (0.2-0.3)  
   - Supporting attributes (categories, types) → lower weight (0.1-0.2)
   - SKIP: IDs, coordinates alone, metadata columns

3. LOCATION DATA: If this is a location-based entity (restaurant, store, venue):
   - Are there latitude/longitude columns in BOTH datasets?
   - If yes: geo_config.enabled=true, weight=0.2-0.3 (location matters!)
   - threshold_meters: 50-100 for urban, 200-500 for rural/sparse data

4. BUCKET THRESHOLDS: These determine how candidates are categorized by similarity score.
   Since blocking already pre-filters for similar records, thresholds should be HIGH:
   - "high" threshold: 0.85-0.95 (near-perfect matches, very likely true matches)
   - "mid" threshold: 0.65-0.80 (moderate similarity, worth checking)
   - Pairs below "mid" go to "low" bucket (likely non-matches)

Now provide your analysis and JSON:

REASONING:
[Explain your choices in 2-3 sentences: what entity type, which columns matter most and why, whether geo matters]

JSON:
{{
    "entity_type": "...",
    "blocking_columns": ["1-2 columns for blocking"],
    "blocker_type": "token|standard|sorted_neighbourhood|embedding",
    "blocker_config": {{"reason": "...", "column": "..."}},
    "match_criteria": "what confirms a match",
    "similarity_columns": [
        {{"column": "...", "function": "...", "weight": 0.X}}
    ],
    "geo_config": {{"enabled": true/false, "weight": 0.X, "threshold_meters": N}},
    "bucket_thresholds": {{"high": 0.X, "mid": 0.X}}
}}

IMPORTANT RULES:
- similarity_columns weights should sum to ~1.0 (geo weight is ADDITIONAL)
- If geo_config.enabled=true, geo weight MUST be > 0 (typically 0.4-0.6)
- Only include columns that exist in BOTH datasets
- Focus on 2-4 most discriminative columns, not every column
- bucket_thresholds.high should be >= 0.85, bucket_thresholds.mid should be >= 0.65"""

        try:
            response = self.llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=human_prompt)])
            response_text = response.content if hasattr(response, 'content') else str(response)
            response_text = response_text.strip()
            
            # Extract JSON from response (may have reasoning before it)
            json_start = response_text.find('{')
            json_end = response_text.rfind('}') + 1
            if json_start != -1 and json_end > json_start:
                json_text = response_text[json_start:json_end]
                analysis = json.loads(json_text)
            else:
                raise ValueError("No JSON found in response")
            
            # Print reasoning if present
            if self.verbose and json_start > 0:
                reasoning = response_text[:json_start].strip()
                if reasoning:
                    print(f"   LLM Reasoning: {reasoning[:200]}...")
            
            if self.verbose:
                print(f"   Entity type: {analysis.get('entity_type', 'unknown')}")
                print(f"   Blocker type: {analysis.get('blocker_type', 'token')}")
                print(f"   Similarity columns:")
                for sc in analysis.get('similarity_columns', []):
                    print(f"      - {sc['column']}: {sc['function']} (weight={sc.get('weight', 1.0)})")
                geo = analysis.get('geo_config', {})
                if geo.get('enabled'):
                    print(f"   Geo scoring: enabled (weight={geo.get('weight')}, threshold={geo.get('threshold_meters')}m)")
                thresholds = analysis.get('bucket_thresholds', {})
                print(f"   Bucket thresholds: high>={thresholds.get('high', 0.85)}, mid>={thresholds.get('mid', 0.65)}")
            return analysis
        except Exception as e:
            if self.verbose:
                print(f"   LLM schema analysis failed: {e}")
            common_cols = list(set(df_a.columns) & set(df_b.columns))
            text_cols = [c for c in common_cols if df_a[c].dtype == 'object']
            # Build fallback similarity_columns
            fallback_sim_cols = []
            cols_to_use = text_cols[:3] if text_cols else common_cols[:3]
            weight_each = 1.0 / len(cols_to_use) if cols_to_use else 1.0
            for col in cols_to_use:
                fallback_sim_cols.append({"column": col, "function": "jaro_winkler", "weight": weight_each})
            return {
                "entity_type": "unknown entity",
                "blocking_columns": text_cols[:1] if text_cols else common_cols[:1],
                "blocker_type": "token",
                "blocker_config": {"reason": "fallback", "column": text_cols[0] if text_cols else common_cols[0]},
                "match_criteria": "Records match if primary attributes are similar",
                "similarity_columns": fallback_sim_cols,
                "geo_config": {"enabled": False, "weight": 0.0, "threshold_meters": 100},
                "bucket_thresholds": {"high": 0.85, "mid": 0.65}
            }
    
    def _normalize_text(self, s) -> str:
        if s is None or (isinstance(s, float) and pd.isna(s)):
            return ""
        return str(s).lower().strip()
    
    def _preprocess_for_blocker(self, value: str) -> str:
        if value is None or (isinstance(value, float) and pd.isna(value)):
            return ""
        return str(value).lower().strip()
    
    def _compute_similarity(self, val_a, val_b, method: str = "jaro_winkler", **kwargs) -> float:
        """Compute similarity using the specified method."""
        a_norm = self._normalize_text(val_a)
        b_norm = self._normalize_text(val_b)
        
        if method == "exact_match":
            return 1.0 if a_norm == b_norm and a_norm else 0.0
        
        if method == "contains":
            if not a_norm or not b_norm:
                return 0.0
            return 1.0 if (a_norm in b_norm or b_norm in a_norm) else 0.0
        
        if method == "numeric_diff":
            try:
                a_num = float(val_a) if val_a is not None else None
                b_num = float(val_b) if val_b is not None else None
                if a_num is None or b_num is None or pd.isna(a_num) or pd.isna(b_num):
                    return 0.0
                if a_num == 0 and b_num == 0:
                    return 1.0
                max_val = max(abs(a_num), abs(b_num))
                if max_val == 0:
                    return 1.0
                return max(0, 1 - abs(a_num - b_num) / max_val)
            except (ValueError, TypeError):
                return 0.0
        
        if method == "token_overlap":
            if not a_norm or not b_norm:
                return 0.0
            tokens_a = set(a_norm.split())
            tokens_b = set(b_norm.split())
            if not tokens_a or not tokens_b:
                return 0.0
            intersection = len(tokens_a & tokens_b)
            union = len(tokens_a | tokens_b)
            return intersection / union if union > 0 else 0.0
        
        if method == "levenshtein":
            if not a_norm or not b_norm:
                return 0.0
            try:
                import textdistance
                return textdistance.levenshtein.normalized_similarity(a_norm, b_norm)
            except ImportError:
                from difflib import SequenceMatcher
                return SequenceMatcher(None, a_norm, b_norm).ratio()
        
        # Default: jaro_winkler
        if not a_norm or not b_norm:
            return 0.0
        try:
            import textdistance
            return textdistance.jaro_winkler(a_norm, b_norm)
        except ImportError:
            from difflib import SequenceMatcher
            return SequenceMatcher(None, a_norm, b_norm).ratio()
    
    def _haversine_km(self, lat1, lon1, lat2, lon2) -> float:
        try:
            if None in (lat1, lon1, lat2, lon2) or any(pd.isna(x) for x in [lat1, lon1, lat2, lon2]):
                return float("inf")
            rlat1, rlon1, rlat2, rlon2 = map(math.radians, [float(lat1), float(lon1), float(lat2), float(lon2)])
            dlat, dlon = rlat2 - rlat1, rlon2 - rlon1
            a = math.sin(dlat/2)**2 + math.cos(rlat1)*math.cos(rlat2)*math.sin(dlon/2)**2
            return 6371.0088 * (2 * math.asin(math.sqrt(a)))
        except:
            return float("inf")
    
    def _compute_geo_similarity(self, lat1, lon1, lat2, lon2, threshold_m: float = 100) -> float:
        """Compute geo similarity - 1.0 if within threshold, decays to 0."""
        dist_km = self._haversine_km(lat1, lon1, lat2, lon2)
        if dist_km == float('inf'):
            return 0.0
        dist_m = dist_km * 1000
        if dist_m <= threshold_m:
            return 1.0
        # Linear decay from threshold to 10x threshold
        max_dist = threshold_m * 10
        if dist_m >= max_dist:
            return 0.0
        return 1 - (dist_m - threshold_m) / (max_dist - threshold_m)
    
    def _create_pydi_blocker(self, df_a: pd.DataFrame, df_b: pd.DataFrame, id_col_a: str, id_col_b: str, schema_analysis: Dict[str, Any]):
        blocker_type = schema_analysis.get('blocker_type', 'token')
        blocker_config = schema_analysis.get('blocker_config', {})
        blocking_cols = schema_analysis.get('blocking_columns', [])
        
        df_left = df_a.copy()
        df_right = df_b.copy()
        df_left['_id'] = df_left[id_col_a].astype(str)
        df_right['_id'] = df_right[id_col_b].astype(str)
        id_column = '_id'
        
        if self.verbose:
            print(f"   Creating PyDI {blocker_type} blocker...")
        
        try:
            if blocker_type == "token":
                block_col = blocker_config.get('column', blocking_cols[0] if blocking_cols else None)
                if block_col is None or block_col not in df_left.columns or block_col not in df_right.columns:
                    for col in blocking_cols:
                        if col in df_left.columns and col in df_right.columns:
                            block_col = col
                            break
                if block_col is None:
                    raise ValueError("No valid blocking column found")
                blocker = TokenBlocker(df_left=df_left, df_right=df_right, column=block_col, id_column=id_column,
                                       min_token_len=self.min_token_len, preprocess=self._preprocess_for_blocker)
            elif blocker_type == "standard":
                on_cols = [c for c in blocking_cols if c in df_left.columns and c in df_right.columns]
                if not on_cols:
                    raise ValueError("No valid columns for StandardBlocker")
                blocker = StandardBlocker(df_left=df_left, df_right=df_right, on=on_cols, id_column=id_column,
                                         preprocess=self._preprocess_for_blocker)
            elif blocker_type == "sorted_neighbourhood":
                key_col = blocker_config.get('column', blocking_cols[0] if blocking_cols else None)
                if key_col is None or key_col not in df_left.columns or key_col not in df_right.columns:
                    for col in blocking_cols:
                        if col in df_left.columns and col in df_right.columns:
                            key_col = col
                            break
                window = blocker_config.get('window_size', 5)
                blocker = SortedNeighbourhoodBlocker(df_left=df_left, df_right=df_right, key=key_col, id_column=id_column,
                                                    window=window, preprocess=self._preprocess_for_blocker)
            elif blocker_type == "embedding":
                text_cols = [c for c in blocking_cols if c in df_left.columns and c in df_right.columns]
                if not text_cols:
                    text_cols = blocking_cols[:1]
                blocker = EmbeddingBlocker(df_left=df_left, df_right=df_right, text_cols=text_cols, id_column=id_column,
                                          model='sentence-transformers/all-MiniLM-L6-v2', top_k=50, threshold=0.3)
            else:
                block_col = blocking_cols[0] if blocking_cols else list(df_left.columns)[0]
                blocker = TokenBlocker(df_left=df_left, df_right=df_right, column=block_col, id_column=id_column,
                                       min_token_len=self.min_token_len, preprocess=self._preprocess_for_blocker)
            return blocker, df_left, df_right, id_column
        except Exception as e:
            if self.verbose:
                print(f"   Failed to create {blocker_type} blocker: {e}, falling back to TokenBlocker")
            block_col = None
            for col in blocking_cols:
                if col in df_left.columns and col in df_right.columns:
                    block_col = col
                    break
            if block_col is None:
                block_col = list(set(df_left.columns) & set(df_right.columns) - {'_id'})[0]
            blocker = TokenBlocker(df_left=df_left, df_right=df_right, column=block_col, id_column=id_column,
                                   min_token_len=self.min_token_len, preprocess=self._preprocess_for_blocker)
            return blocker, df_left, df_right, id_column
    
    def _build_candidates(self, df_a: pd.DataFrame, df_b: pd.DataFrame, id_col_a: str, id_col_b: str, schema_analysis: Dict[str, Any]) -> pd.DataFrame:
        """Build candidates using LLM-specified similarity functions and weights."""
        similarity_columns = schema_analysis.get('similarity_columns', [])
        geo_config = schema_analysis.get('geo_config', {'enabled': False})
        bucket_thresholds = schema_analysis.get('bucket_thresholds', {'high': 0.85, 'mid': 0.65})
        
        # Fallback to old format if needed
        if not similarity_columns and 'comparator_columns' in schema_analysis:
            similarity_columns = [{"column": c, "function": "jaro_winkler", "weight": 1.0} 
                                  for c in schema_analysis['comparator_columns']]
        
        blocker, df_left, df_right, id_column = self._create_pydi_blocker(df_a, df_b, id_col_a, id_col_b, schema_analysis)
        self._blocker_instance = blocker
        
        try:
            candidate_pairs = blocker.materialize()
            if self.verbose:
                print(f"   Materialized {len(candidate_pairs):,} candidate pairs")
            if len(candidate_pairs) > self.max_candidates:
                print(f"    TRUNCATING: {len(candidate_pairs):,} -> {self.max_candidates:,} (proportional sample)")
                candidate_pairs = candidate_pairs.sample(n=self.max_candidates, random_state=42)
        except Exception as e:
            if self.verbose:
                print(f"   Blocker materialization failed: {e}")
            return pd.DataFrame()
        
        if candidate_pairs.empty:
            return pd.DataFrame()
        
        cand = candidate_pairs.copy()
        cand = cand.rename(columns={'id1': 'id_a', 'id2': 'id_b'})
        
        # Store all candidate IDs for random negative generation
        self._all_candidate_ids = set(zip(cand['id_a'].astype(str), cand['id_b'].astype(str)))
        
        df_left_indexed = df_left.set_index('_id')
        df_right_indexed = df_right.set_index('_id')
        
        for col in df_a.columns:
            if col != id_col_a:
                cand[f'{col}_a'] = cand['id_a'].map(lambda x: df_left_indexed.loc[x, col] if x in df_left_indexed.index else None)
        for col in df_b.columns:
            if col != id_col_b:
                cand[f'{col}_b'] = cand['id_b'].map(lambda x: df_right_indexed.loc[x, col] if x in df_right_indexed.index else None)
        
        # Apply LLM-specified similarity functions with weights
        if self.verbose and similarity_columns:
            print(f"   Computing similarity scores...")
        
        weighted_scores = []
        total_weight = 0
        
        for sim_config in similarity_columns:
            col = sim_config['column']
            func = sim_config.get('function', 'jaro_winkler')
            weight = sim_config.get('weight', 1.0)
            col_a, col_b = f'{col}_a', f'{col}_b'
            
            if col_a in cand.columns and col_b in cand.columns:
                sim_col = f'sim_{col}'
                cand[sim_col] = cand.apply(
                    lambda r: self._compute_similarity(r.get(col_a), r.get(col_b), method=func), 
                    axis=1
                )
                weighted_scores.append((sim_col, weight))
                total_weight += weight
        
        # Compute geo distance if present
        has_geo = 'latitude_a' in cand.columns and 'longitude_a' in cand.columns
        if has_geo:
            cand['geo_km'] = cand.apply(lambda r: self._haversine_km(r.get('latitude_a'), r.get('longitude_a'), r.get('latitude_b'), r.get('longitude_b')), axis=1)
            cand['geo_m'] = (cand['geo_km'] * 1000).round(1)
            
            # Add geo similarity if enabled by LLM
            if geo_config.get('enabled', False):
                threshold_m = geo_config.get('threshold_meters', 100)
                geo_weight = geo_config.get('weight', 0.3)
                cand['sim_geo'] = cand.apply(
                    lambda r: self._compute_geo_similarity(
                        r.get('latitude_a'), r.get('longitude_a'),
                        r.get('latitude_b'), r.get('longitude_b'),
                        threshold_m=threshold_m
                    ), axis=1
                )
                weighted_scores.append(('sim_geo', geo_weight))
                total_weight += geo_weight
                if self.verbose:
                    print(f"   Geo scoring enabled: threshold={threshold_m}m, weight={geo_weight}")
        
        # Compute weighted score
        if weighted_scores and total_weight > 0:
            cand['score'] = sum(cand[col] * weight for col, weight in weighted_scores) / total_weight
            cand['score'] = cand['score'].round(4)
        else:
            cand['score'] = 0.5
        
        # Apply LLM-specified bucket thresholds
        high_thresh = bucket_thresholds.get('high', 0.85)
        mid_thresh = bucket_thresholds.get('mid', 0.65)
        
        cand['bucket'] = 'low'
        cand.loc[cand['score'] >= high_thresh, 'bucket'] = 'high'
        cand.loc[(cand['score'] >= mid_thresh) & (cand['score'] < high_thresh), 'bucket'] = 'mid'
        cand = cand.drop_duplicates(subset=['id_a', 'id_b'])
        
        if self.verbose:
            print(f"   Bucket thresholds: high>={high_thresh}, mid>={mid_thresh}")
        
        return cand.sort_values('score', ascending=False).reset_index(drop=True)
    
    def _format_record_for_prompt(self, row: pd.Series, suffix: str) -> str:
        lines = []
        exclude_prefixes = ['_block', 'sim_', 'score', 'bucket', 'label', 'geo_km', 'geo_m']
        for col in sorted(row.index):
            if col.endswith(f"_{suffix}"):
                if any(col.startswith(p) for p in exclude_prefixes):
                    continue
                base_col = col.replace(f"_{suffix}", "")
                if base_col in ['id']:
                    continue
                val = row[col]
                if pd.notna(val) and str(val).strip():
                    lines.append(f"  {base_col}: {val}")
        return "\n".join(lines) if lines else "  (no attributes available)"
    
    def _label_single(self, row: pd.Series, idx: int, total: int, schema_analysis: Dict[str, Any]) -> int:
        entity_type = schema_analysis.get('entity_type', 'entity')
        
        system_prompt = f"""You are an expert at matching {entity_type} records.
Label 1 if records represent the SAME real-world entity.
Label 0 if records are CLEARLY DIFFERENT entities.

IMPORTANT:
- Minor formatting differences are OK (capitalization, punctuation, date formats like "2004" vs "2004-01-01")
- Missing or zero values should NOT prevent a match if other key fields align
- Focus on core identifying attributes (names, creators, dates)
- If the key identifiers match, label as 1

RESPOND WITH ONLY: 1 or 0"""

        record_a = self._format_record_for_prompt(row, 'a')
        record_b = self._format_record_for_prompt(row, 'b')
        score = row.get('score', 0)
        
        human_prompt = f"""Pair [{idx}/{total}]

RECORD A:
{record_a}

RECORD B:
{record_b}

Similarity score: {score:.2f}

Are these the SAME {entity_type}? Reply 1 or 0:"""

        try:
            response = self.llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=human_prompt)])
            response_text = (response.content if hasattr(response, 'content') else str(response)).strip()
            if '1' in response_text[:5]:
                return 1
            return 0
        except Exception as e:
            if self.verbose:
                print(f"   Label error: {e}")
            return 0
    
    def _label_batch(self, batch_df: pd.DataFrame, schema_analysis: Dict[str, Any]) -> List[int]:
        entity_type = schema_analysis.get('entity_type', 'entity')
        
        system_prompt = f"""You are an expert at matching {entity_type} records.
Label 1 if records represent the SAME real-world entity.
Label 0 if records are CLEARLY DIFFERENT entities.

IMPORTANT:
- Minor formatting differences are OK (capitalization, punctuation, date formats)
- Missing or zero values should NOT prevent a match if other key fields align
- Focus on core identifying attributes

Return a JSON array of integers. Example: [1, 0, 0]"""

        pairs_text = []
        for idx, (_, row) in enumerate(batch_df.iterrows()):
            record_a = self._format_record_for_prompt(row, 'a')
            record_b = self._format_record_for_prompt(row, 'b')
            score = row.get('score', 0)
            pairs_text.append(f"--- PAIR {idx+1} (score={score:.2f}) ---\nA:\n{record_a}\nB:\n{record_b}\n")
        
        human_prompt = f"""Label {len(batch_df)} pairs:\n{"".join(pairs_text)}\nReturn JSON array with {len(batch_df)} integers:"""

        try:
            response = self.llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=human_prompt)])
            response_text = (response.content if hasattr(response, 'content') else str(response)).strip()
            if response_text.startswith("```"):
                response_text = "\n".join([l for l in response_text.split("\n") if not l.strip().startswith("```")])
            labels = json.loads(response_text)
            if isinstance(labels, list) and len(labels) == len(batch_df):
                return [int(l) for l in labels]
        except:
            pass
        return [self._label_single(row, i, len(batch_df), schema_analysis) for i, (_, row) in enumerate(batch_df.iterrows())]
    
    def _compute_score_for_row(self, row: Dict, schema_analysis: Dict[str, Any]) -> float:
        """Compute score for a single row using schema_analysis config."""
        similarity_columns = schema_analysis.get('similarity_columns', [])
        geo_config = schema_analysis.get('geo_config', {'enabled': False})
        
        weighted_scores = []
        total_weight = 0
        
        for sim_config in similarity_columns:
            col = sim_config['column']
            func = sim_config.get('function', 'jaro_winkler')
            weight = sim_config.get('weight', 1.0)
            col_a, col_b = f'{col}_a', f'{col}_b'
            
            if col_a in row and col_b in row:
                sim = self._compute_similarity(row.get(col_a), row.get(col_b), method=func)
                weighted_scores.append(sim * weight)
                total_weight += weight
        
        # Add geo if enabled
        if geo_config.get('enabled', False):
            lat_a, lon_a = row.get('latitude_a'), row.get('longitude_a')
            lat_b, lon_b = row.get('latitude_b'), row.get('longitude_b')
            if all(v is not None for v in [lat_a, lon_a, lat_b, lon_b]):
                threshold_m = geo_config.get('threshold_meters', 100)
                geo_weight = geo_config.get('weight', 0.3)
                geo_sim = self._compute_geo_similarity(lat_a, lon_a, lat_b, lon_b, threshold_m)
                weighted_scores.append(geo_sim * geo_weight)
                total_weight += geo_weight
        
        return sum(weighted_scores) / total_weight if total_weight > 0 else 0.0
    
    def _generate_random_negatives(self, n_needed: int, used_pairs: set, schema_analysis: Dict[str, Any], seed: int = 42) -> pd.DataFrame:
        """Generate random pairs NOT in blocker output - guaranteed non-matches."""
        if self._df_a is None or self._df_b is None:
            return pd.DataFrame()
        
        np.random.seed(seed)
        ids_a = self._df_a[self._id_col_a].astype(str).tolist()
        ids_b = self._df_b[self._id_col_b].astype(str).tolist()
        blocker_pairs = self._all_candidate_ids or set()
        
        random_pairs = []
        attempts = 0
        max_attempts = n_needed * 100
        
        while len(random_pairs) < n_needed and attempts < max_attempts:
            id_a = np.random.choice(ids_a)
            id_b = np.random.choice(ids_b)
            pair = (id_a, id_b)
            if pair not in blocker_pairs and pair not in used_pairs:
                random_pairs.append(pair)
                used_pairs.add(pair)
            attempts += 1
        
        if not random_pairs:
            return pd.DataFrame()
        
        df_a_indexed = self._df_a.set_index(self._id_col_a)
        df_b_indexed = self._df_b.set_index(self._id_col_b)
        
        rows = []
        for id_a, id_b in random_pairs:
            row = {'id_a': id_a, 'id_b': id_b}
            if id_a in df_a_indexed.index:
                for col in self._df_a.columns:
                    if col != self._id_col_a:
                        row[f'{col}_a'] = df_a_indexed.loc[id_a, col]
            if id_b in df_b_indexed.index:
                for col in self._df_b.columns:
                    if col != self._id_col_b:
                        row[f'{col}_b'] = df_b_indexed.loc[id_b, col]
            
            # Use the same scoring logic as _build_candidates
            row['score'] = self._compute_score_for_row(row, schema_analysis)
            row['bucket'] = 'random_negative'
            rows.append(row)
        
        return pd.DataFrame(rows)
    
    def _rebalance_matches(self, samples: pd.DataFrame, schema_analysis: Dict[str, Any], seed: int = 42) -> pd.DataFrame:
        target_matches = int(round(self.n_samples * self.target_match_pct))
        max_matches = int(round(self.n_samples * (self.target_match_pct + 0.10)))
        current_matches = int((samples['label'] == 1).sum())
        
        print(f"\nREBALANCING CHECK:")
        print(f"   Target: {target_matches} matches (max {max_matches})")
        print(f"   Current: {current_matches} matches, {len(samples) - current_matches} non-matches")
        
        if current_matches < target_matches:
            return self._add_more_matches(samples, schema_analysis, target_matches, seed)
        if current_matches > max_matches:
            return self._add_more_non_matches(samples, schema_analysis, target_matches, seed)
        
        print(f"Already balanced")
        return samples
    
    def _add_more_matches(self, samples: pd.DataFrame, schema_analysis: Dict[str, Any], target_matches: int, seed: int) -> pd.DataFrame:
        current_matches = int((samples['label'] == 1).sum())
        
        # Try high candidates first, then mid candidates
        reserve = None
        if self._reserve_high_candidates is not None and not self._reserve_high_candidates.empty:
            reserve = self._reserve_high_candidates.copy()
            reserve_type = "high"
        elif self._reserve_mid_candidates is not None and not self._reserve_mid_candidates.empty:
            reserve = self._reserve_mid_candidates.copy()
            reserve_type = "mid"
        
        if reserve is None or reserve.empty:
            print(f"   No reserve candidates for adding matches")
            return samples
        
        used_pairs = set(zip(samples['id_a'].astype(str), samples['id_b'].astype(str)))
        reserve = reserve[~reserve.apply(lambda r: (str(r['id_a']), str(r['id_b'])) in used_pairs, axis=1)]
        
        # Sort by score descending - check highest similarity first
        if 'score' in reserve.columns:
            reserve = reserve.sort_values('score', ascending=False)
        
        print(f"   ADDING MATCHES: Need {target_matches - current_matches} more")
        print(f"   Reserve: {len(reserve)} {reserve_type}-prob candidates (sorted by score)")
        
        total_added = 0
        total_checked = 0
        round_num = 0
        max_rounds = 50  # Much more aggressive
        batch_size = 50  # Bigger batches
        
        while current_matches < target_matches and round_num < max_rounds and not reserve.empty:
            round_num += 1
            actual_batch_size = min(batch_size, len(reserve))
            
            # Take from top (highest scores) instead of random
            batch = reserve.head(actual_batch_size).copy()
            reserve = reserve.iloc[actual_batch_size:]
            
            batch['label'] = self._label_batch(batch, schema_analysis)
            new_matches = batch[batch['label'] == 1]
            total_checked += len(batch)
            
            print(f"   Round {round_num}: Checked {len(batch)}, found {len(new_matches)} matches (total checked: {total_checked})")
            
            if len(new_matches) > 0:
                n_to_add = min(len(new_matches), target_matches - current_matches)
                non_matches = samples[samples['label'] == 0].sort_values('score', ascending=True)
                n_to_replace = min(n_to_add, len(non_matches))
                if n_to_replace > 0:
                    samples = samples.drop(non_matches.head(n_to_replace).index)
                    samples = pd.concat([samples, new_matches.head(n_to_replace)], ignore_index=True)
                    total_added += n_to_replace
                    current_matches = int((samples['label'] == 1).sum())
                    print(f"   ✓ Added {n_to_replace} -> Now {current_matches}/{target_matches}")
        
        print(f"   Total: checked {total_checked}, added {total_added}")
        return samples.reset_index(drop=True)
    
    def _add_more_non_matches(self, samples: pd.DataFrame, schema_analysis: Dict[str, Any], target_matches: int, seed: int) -> pd.DataFrame:
        """Replace excess matches with confirmed non-matches to hit target ratio."""
        current_matches = int((samples['label'] == 1).sum())
        excess = current_matches - target_matches
        used_pairs = set(zip(samples['id_a'].astype(str), samples['id_b'].astype(str)))
        
        print(f"\n   REDUCING MATCHES: Have {current_matches} matches, need {target_matches} (removing {excess})")
        
        total_replaced = 0
        round_num = 0
        
        while current_matches > target_matches and round_num < self.max_resample_rounds * 3:
            round_num += 1
            excess_now = current_matches - target_matches
            
            batch = self._generate_random_negatives(
                n_needed=min(self.resample_batch_size, excess_now * 3),
                used_pairs=used_pairs,
                schema_analysis=schema_analysis,
                seed=seed + round_num * 1000
            )
            
            if batch.empty:
                print(f"   Round {round_num}: Cannot generate more random pairs")
                break
            
            print(f"   Round {round_num}: Labeling {len(batch)} random pairs...")
            batch = batch.copy()
            batch['label'] = self._label_batch(batch, schema_analysis)
            
            new_non_matches = batch[batch['label'] == 0]
            print(f"      -> {len(new_non_matches)}/{len(batch)} confirmed as non-matches")
            
            if len(new_non_matches) > 0:
                n_to_replace = min(len(new_non_matches), excess_now)
                matches_to_remove = samples[samples['label'] == 1].sort_values('score', ascending=False).head(n_to_replace)
                
                if len(matches_to_remove) > 0:
                    samples = samples.drop(matches_to_remove.index)
                    samples = pd.concat([samples, new_non_matches.head(n_to_replace)], ignore_index=True)
                    total_replaced += n_to_replace
                    current_matches = int((samples['label'] == 1).sum())
                    print(f"      -> Replaced {n_to_replace} high-sim matches -> Now {current_matches}/{target_matches} matches")
                    
                    if current_matches <= target_matches:
                        print(f"   ✓ Target reached!")
                        break
        
        print(f"   Total: Replaced {total_replaced} matches with random non-matches")
        return samples.reset_index(drop=True)
    
    def _get_display_name(self, row: pd.Series, suffix: str, schema_analysis: Dict[str, Any]) -> str:
        # Try similarity_columns first
        for sc in schema_analysis.get('similarity_columns', []):
            col = sc['column']
            col_with_suffix = f"{col}_{suffix}"
            if col_with_suffix in row.index and pd.notna(row[col_with_suffix]):
                return str(row[col_with_suffix])[:40]
        for col in ['name', 'name_norm', 'title', 'label']:
            col_with_suffix = f"{col}_{suffix}"
            if col_with_suffix in row.index and pd.notna(row[col_with_suffix]):
                return str(row[col_with_suffix])[:40]
        return f"ID:{row.get(f'id_{suffix}', '?')}"
    
    def _get_checkpoint_path(self, pair_name: str) -> str:
        return os.path.join(self.output_dir, f"checkpoint_{pair_name}.csv")
    
    def _get_schema_cache_path(self, pair_name: str) -> str:
        return os.path.join(self.output_dir, f"schema_{pair_name}.json")
    
    def _load_checkpoint(self, pair_name: str) -> Tuple[Optional[pd.DataFrame], int]:
        checkpoint_path = self._get_checkpoint_path(pair_name)
        if os.path.exists(checkpoint_path):
            try:
                df = pd.read_csv(checkpoint_path)
                labeled_count = df['label'].notna().sum()
                if self.verbose:
                    print(f"Found checkpoint: {labeled_count} samples labeled")
                return df, int(labeled_count)
            except:
                pass
        return None, 0
    
    def _load_schema_cache(self, pair_name: str) -> Optional[Dict[str, Any]]:
        cache_path = self._get_schema_cache_path(pair_name)
        if os.path.exists(cache_path):
            try:
                with open(cache_path, 'r') as f:
                    schema = json.load(f)
                    # Check if it's new format (has similarity_columns)
                    if 'similarity_columns' in schema:
                        return schema
                    else:
                        if self.verbose:
                            print(f"   Found old schema format, will regenerate...")
                        return None
            except:
                pass
        return None
    
    def _save_schema_cache(self, schema_analysis: Dict[str, Any], pair_name: str):
        try:
            with open(self._get_schema_cache_path(pair_name), 'w') as f:
                json.dump(schema_analysis, f, indent=2)
        except:
            pass
    
    def _save_checkpoint(self, df: pd.DataFrame, pair_name: str, labeled_count: int):
        df.to_csv(self._get_checkpoint_path(pair_name), index=False)
    
    def _stratified_sample(self, candidates: pd.DataFrame, n_total: int, seed: int = 42, preserve_reserve: bool = True) -> pd.DataFrame:
        if candidates.empty:
            return candidates
        if 'bucket' not in candidates.columns:
            candidates['bucket'] = 'low'
        
        n_high = max(1, int(round(n_total * 0.20)))
        n_mid = max(1, int(round(n_total * 0.30)))
        n_low = max(1, n_total - n_high - n_mid)
        
        high = candidates[candidates['bucket'] == 'high']
        mid = candidates[candidates['bucket'] == 'mid']
        low = candidates[candidates['bucket'] == 'low']
        
        if self.verbose:
            print(f"   Buckets: {len(high)} high, {len(mid)} mid, {len(low)} low")
        
        def take(df, n):
            return df.sample(n=min(n, len(df)), random_state=seed) if not df.empty else df
        
        sampled = pd.concat([take(high, n_high), take(mid, n_mid), take(low, n_low)], ignore_index=True)
        
        if len(sampled) < n_total:
            used_ids = set(zip(sampled.get('id_a', []), sampled.get('id_b', [])))
            rest = candidates[~candidates.apply(lambda r: (r.get('id_a'), r.get('id_b')) in used_ids, axis=1)]
            if not rest.empty:
                extra = rest.sample(n=min(n_total - len(sampled), len(rest)), random_state=seed)
                sampled = pd.concat([sampled, extra], ignore_index=True)
        
        if preserve_reserve:
            sampled_ids = set(zip(sampled['id_a'].astype(str), sampled['id_b'].astype(str)))
            self._reserve_high_candidates = high[~high.apply(lambda r: (str(r['id_a']), str(r['id_b'])) in sampled_ids, axis=1)].copy()
            self._reserve_mid_candidates = mid[~mid.apply(lambda r: (str(r['id_a']), str(r['id_b'])) in sampled_ids, axis=1)].copy()
            self._reserve_low_candidates = low[~low.apply(lambda r: (str(r['id_a']), str(r['id_b'])) in sampled_ids, axis=1)].copy()
            if self.verbose:
                print(f"   Reserved: {len(self._reserve_high_candidates)} high, {len(self._reserve_mid_candidates)} mid, {len(self._reserve_low_candidates)} low")
        
        sampled['label'] = pd.NA
        return sampled.head(n_total)
    
    def run(self, df_a: pd.DataFrame, df_b: pd.DataFrame, pair_name: str, id_col_a: str, id_col_b: str) -> pd.DataFrame:
        print(f"\n{'='*60}")
        print(f"BLOCKING TEST SET GENERATOR: {pair_name}")
        print(f"{'='*60}")
        print(f"Target: {self.n_samples} samples, {self.target_match_pct*100:.0f}% matches ({int(self.n_samples * self.target_match_pct)} matches)")
        
        # Store for random negative generation
        self._df_a = df_a
        self._df_b = df_b
        self._id_col_a = id_col_a
        self._id_col_b = id_col_b
        
        # Reset token usage for this run
        self.token_usage = {'prompt_tokens': 0, 'completion_tokens': 0, 'total_tokens': 0, 'total_cost': 0.0}
        
        # Use callback to track token usage (works for OpenAI models)
        with get_openai_callback() as cb:
            result = self._run_internal(pair_name)
            
            # Store token usage
            self.token_usage = {
                'prompt_tokens': cb.prompt_tokens,
                'completion_tokens': cb.completion_tokens,
                'total_tokens': cb.total_tokens,
                'total_cost': cb.total_cost
            }
            
            # Print token usage summary
            print(f"\nTOKEN USAGE:")
            print(f"   Prompt tokens: {cb.prompt_tokens:,}")
            print(f"   Completion tokens: {cb.completion_tokens:,}")
            print(f"   Total tokens: {cb.total_tokens:,}")
            print(f"   Estimated cost: ${cb.total_cost:.4f}")
        
        return result
    
    def _run_internal(self, pair_name: str) -> pd.DataFrame:
        """Internal run method wrapped by token tracking."""
        checkpoint_df, start_idx = self._load_checkpoint(pair_name)
        if checkpoint_df is not None and start_idx >= self.n_samples:
            print(f"Already complete!")
            return checkpoint_df[['id_a', 'id_b', 'label']]
        
        schema_analysis = self._load_schema_cache(pair_name)
        if schema_analysis is None:
            print(f"\nAnalyzing schemas with LLM...")
            schema_analysis = self._analyze_schema_with_llm(self._df_a, self._df_b)
            self._save_schema_cache(schema_analysis, pair_name)
        self._schema_analysis = schema_analysis
        
        if checkpoint_df is None:
            print(f"\nBuilding candidates...")
            candidates = self._build_candidates(self._df_a, self._df_b, self._id_col_a, self._id_col_b, schema_analysis)
            if candidates.empty:
                print("No candidates found!")
                return pd.DataFrame(columns=['id_a', 'id_b', 'label'])
            
            print(f"Total candidates: {len(candidates):,}")
            print(f"Sampling {self.n_samples} candidates...")
            samples = self._stratified_sample(candidates, self.n_samples, preserve_reserve=True)
            
            if 'bucket' in samples.columns:
                print(f"   Sampled: {samples['bucket'].value_counts().to_dict()}")
        else:
            samples = checkpoint_df
        
        print(f"\nLabeling from index {start_idx}...")
        print("-" * 60)
        
        for i in range(start_idx, len(samples)):
            row = samples.iloc[i]
            name_a = self._get_display_name(row, 'a', schema_analysis)
            name_b = self._get_display_name(row, 'b', schema_analysis)
            
            label = self._label_single(row, i + 1, self.n_samples, schema_analysis)
            label_str = "MATCH" if label == 1 else "NO MATCH"
            bucket = row.get('bucket', '?')
            score = row.get('score', 0)
            print(f"[{i+1:3d}/{self.n_samples}] '{name_a}' vs '{name_b}' -> {label_str} (bucket={bucket}, score={score:.2f})")
            
            samples.at[samples.index[i], 'label'] = label
            
            if (i + 1) % self.checkpoint_interval == 0:
                self._save_checkpoint(samples, pair_name, i + 1)
            time.sleep(0.1)
        
        print("-" * 60)
        samples = self._rebalance_matches(samples, schema_analysis)
        
        output_path = os.path.join(self.output_dir, f"{pair_name}_goldstandard_blocking.csv")
        gold_standard = samples[['id_a', 'id_b', 'label']].copy()
        gold_standard['label'] = gold_standard['label'].astype(int)
        gold_standard.to_csv(output_path, index=False)
        
        match_count = (gold_standard['label'] == 1).sum()
        non_match_count = (gold_standard['label'] == 0).sum()
        
        print(f"\nLABELING COMPLETE")
        print(f"   Total: {len(gold_standard)}")
        print(f"   Matches: {match_count} ({match_count/len(gold_standard)*100:.1f}%)")
        print(f"   Non-matches: {non_match_count} ({non_match_count/len(gold_standard)*100:.1f}%)")
        print(f"   Saved to: {output_path}")
        
        samples.to_csv(os.path.join(self.output_dir, f"{pair_name}_detailed.csv"), index=False)
        
        checkpoint_path = self._get_checkpoint_path(pair_name)
        if os.path.exists(checkpoint_path):
            os.remove(checkpoint_path)
        
        return gold_standard


def get_llm(provider: str = "gpt"):
    if provider == "gpt":
        return ChatOpenAI(model="gpt-4o-mini", temperature=0)
    elif provider == "gemini":
        from langchain_google_genai import ChatGoogleGenerativeAI
        return ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)
    elif provider == "groq":
        from langchain_groq import ChatGroq
        return ChatGroq(model="llama-3.3-70b-versatile", temperature=0.2)
    else:
        raise ValueError(f"Unknown provider: {provider}")

In [42]:
df_discogs = pd.read_xml("input/datasets/discogs.xml")
df_musicbrainz = pd.read_xml("input/datasets/musicbrainz.xml")

In [48]:
llm = get_llm("gpt")

testset_agent = BlockingTestSetAgent(
    llm=llm,
    output_dir="output/testset-generation",
    n_samples=20,            # Total samples to label
    checkpoint_interval=10,  # Save every 10 labels
    verbose=True,
    max_candidates=100000,   # Limit candidate pairs to 100K
    min_token_len=4          # Require tokens of length 4+ to reduce common matches
)

gold_standard_dm = testset_agent.run(
    df_a=df_discogs,
    df_b=df_musicbrainz,
    pair_name="discogs_musicbrainz",
    id_col_a="id",
    id_col_b="id"
)


BLOCKING TEST SET GENERATOR: discogs_musicbrainz
Target: 20 samples, 20% matches (4 matches)

Analyzing schemas with LLM...
   LLM Reasoning: REASONING:
The records in these datasets represent music releases, specifically songs or albums. The most important columns for matching are "name" and "artist," as they are primary identifiers that d...
   Entity type: music_release
   Blocker type: standard
   Similarity columns:
      - name: jaro_winkler (weight=0.4)
      - artist: jaro_winkler (weight=0.3)
      - release-date: exact_match (weight=0.2)
      - release-country: exact_match (weight=0.1)
   Bucket thresholds: high>=0.85, mid>=0.65

Building candidates...
   Creating PyDI standard blocker...
   Materialized 2,183 candidate pairs
   Computing similarity scores...
   Bucket thresholds: high>=0.85, mid>=0.65
Total candidates: 2,183
Sampling 20 candidates...
   Buckets: 570 high, 1613 mid, 0 low
   Reserved: 562 high, 1601 mid, 0 low
   Sampled: {'mid': 12, 'high': 8}

Labeling fr

In [38]:
df_kaggle = pd.read_parquet('input/datasets/kaggle_small.parquet')
df_yelp = pd.read_parquet('input/datasets/yelp_small.parquet')

In [49]:
llm = get_llm("gpt")

testset_agent_restaurants = BlockingTestSetAgent(
    llm=llm,
    output_dir="output/testset-generation",
    n_samples=30,            # Total samples to label
    checkpoint_interval=10,  # Save every 10 labels
    verbose=True,
    max_candidates=100000,   # Limit candidate pairs to 100K
    min_token_len=3          # Require tokens of length 3+ (restaurants have shorter words)
)

# Run on Kaggle vs Yelp
gold_standard_ky = testset_agent_restaurants.run(
    df_a=df_kaggle,
    df_b=df_yelp,
    pair_name="kaggle_yelp",
    id_col_a="kaggle380k_id",
    id_col_b="yelp_id"
)


BLOCKING TEST SET GENERATOR: kaggle_yelp
Target: 30 samples, 20% matches (6 matches)

Analyzing schemas with LLM...
   LLM Reasoning: REASONING:
The records in both datasets represent restaurants, which are entities that can be matched based on their names, addresses, and other attributes. The most important columns for matching are...
   Entity type: restaurant
   Blocker type: token
   Similarity columns:
      - name_norm: jaro_winkler (weight=0.4)
      - address_line1: exact_match (weight=0.3)
      - city: exact_match (weight=0.1)
      - postal_code: exact_match (weight=0.1)
   Geo scoring: enabled (weight=0.2, threshold=100m)
   Bucket thresholds: high>=0.85, mid>=0.7

Building candidates...
   Creating PyDI token blocker...
   Materialized 2,735,671 candidate pairs
   ⚠️  TRUNCATING: 2,735,671 -> 100,000 (proportional sample)
   Computing similarity scores...
   Geo scoring enabled: threshold=100m, weight=0.2
   Bucket thresholds: high>=0.85, mid>=0.7
Total candidates: 100,00

In [52]:
fullcontact = pd.read_xml('input/datasets/fullcontact.xml')
dpedia = pd.read_xml('input/datasets/dbpedia.xml')

In [54]:
llm = get_llm("gpt")

testset_agent_restaurants = BlockingTestSetAgent(
    llm=llm,
    output_dir="output/testset-generation",
    n_samples=30,            # Total samples to label
    checkpoint_interval=10,  # Save every 10 labels
    verbose=True,
    max_candidates=100000,   # Limit candidate pairs to 100K
    min_token_len=3          # Require tokens of length 3+ (restaurants have shorter words)
)

# Run on Kaggle vs Yelp
gold_standard_ky = testset_agent_restaurants.run(
    df_a=fullcontact,
    df_b=dpedia,
    pair_name="dpedia_fullcontact",
    id_col_a="id",
    id_col_b="id"
)


BLOCKING TEST SET GENERATOR: dpedia_fullcontact
Target: 30 samples, 20% matches (6 matches)

Analyzing schemas with LLM...
   LLM Reasoning: REASONING:
The records in these datasets represent companies or organizations, as indicated by the presence of columns like "name," "country," "city," and "founded." The most important columns for mat...
   Entity type: company
   Blocker type: token
   Similarity columns:
      - name: jaro_winkler (weight=0.4)
      - country: exact_match (weight=0.3)
      - city: token_overlap (weight=0.2)
      - founded: numeric_diff (weight=0.1)
   Bucket thresholds: high>=0.85, mid>=0.65

Building candidates...
   Creating PyDI token blocker...
   Materialized 167,526 candidate pairs
   ⚠️  TRUNCATING: 167,526 -> 100,000 (proportional sample)
   Computing similarity scores...
   Bucket thresholds: high>=0.85, mid>=0.65
Total candidates: 100,000
Sampling 30 candidates...
   Buckets: 64 high, 672 mid, 99264 low
   Reserved: 58 high, 663 mid, 99249 low
   Sa

In [56]:
sales= pd.read_xml('input/datasets/sales.xml')
metacritic = pd.read_xml('input/datasets/metacritic.xml')
llm = get_llm("gpt")

testset_agent_restaurants = BlockingTestSetAgent(
    llm=llm,
    output_dir="output/testset-generation",
    n_samples=30,            # Total samples to label
    checkpoint_interval=10,  # Save every 10 labels
    verbose=True,
    max_candidates=100000,   # Limit candidate pairs to 100K
    min_token_len=3          # Require tokens of length 3+ (restaurants have shorter words)
)

gold_standard_ky = testset_agent_restaurants.run(
    df_a=sales,
    df_b=metacritic,
    pair_name="metacritic_sales",
    id_col_a="id",
    id_col_b="id"
)


BLOCKING TEST SET GENERATOR: metacritic_sales
Target: 30 samples, 20% matches (6 matches)

Analyzing schemas with LLM...
   LLM Reasoning: **REASONING:**
The records in both datasets represent video games, which are identified by their titles, release years, developers, and platforms. The most important columns for matching are the `name...
   Entity type: video_game
   Blocker type: token
   Similarity columns:
      - name: jaro_winkler (weight=0.4)
      - releaseYear: exact_match (weight=0.3)
      - developer: exact_match (weight=0.2)
      - platform: exact_match (weight=0.1)
   Bucket thresholds: high>=0.85, mid>=0.65

Building candidates...
   Creating PyDI token blocker...
   Materialized 5,850,579 candidate pairs
   ⚠️  TRUNCATING: 5,850,579 -> 100,000 (proportional sample)
   Computing similarity scores...
   Bucket thresholds: high>=0.85, mid>=0.65
Total candidates: 100,000
Sampling 30 candidates...
   Buckets: 219 high, 389 mid, 99392 low
   Reserved: 213 high, 380 mid, 9